# Module 03 Lab — Research Agent with Persistent Memory

**Course:** AI Agents Teaching Platform  
**Module:** 03 — Memory & Knowledge  
**Estimated time:** 2–3 hours

---

### What you'll build

A research agent with long-term memory backed by markdown notes and a ChromaDB vector index.
The agent checks memory before synthesising, writes new knowledge as linked notes, and runs
a consolidation pass that archives stale low-importance notes.

### Hard constraints
- Agent must check vault memory before any synthesis
- Every note must include at least one `[[backlink]]`
- Consolidation must reduce note count (archive low-importance notes)
- All memory operations must be logged

> **VS Code / local?** See the lab page on the platform for instructions.

## Step 1 — Choose your model

| Provider | Model string | Key env var |
|---|---|---|
| Anthropic | `claude-haiku-4-5-20251001` | `ANTHROPIC_API_KEY` |
| OpenAI | `gpt-4o-mini` | `OPENAI_API_KEY` |
| Google Gemini | `gemini/gemini-1.5-flash` | `GEMINI_API_KEY` |
| Groq | `groq/llama-3.1-8b-instant` | `GROQ_API_KEY` |

In [ ]:
MODEL = "claude-haiku-4-5-20251001"
print(f"Model: {MODEL}")

In [ ]:
%pip install -q litellm chromadb pyyaml

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your API key: ")
print("Key set ✓")

os.makedirs("vault/notes", exist_ok=True)
os.makedirs("vault/archive", exist_ok=True)
print("Vault directories created ✓")

## Step 4 — Build vault memory (TODO)

Implement four functions:

**`write_note(title, body, tags, links)`** — builds frontmatter + body + `## Related` section, writes to `vault/notes/<slug>.md`

**`read_note(title)`** — reads the note, increments `access_count`, returns a `Note` or `None`

**`sync_embeddings()`** — upserts changed notes to ChromaDB using SHA-256 hash comparison, deletes stale entries

**`search_similar(query, k=4)`** — queries ChromaDB, returns matching Notes

In [ ]:
from pathlib import Path
from datetime import date
from dataclasses import dataclass
import yaml
import re
import hashlib
import logging
import chromadb

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)

VAULT   = Path("vault/notes")
ARCHIVE = Path("vault/archive")
DB_PATH = "./vault_index"

@dataclass
class Note:
    title: str
    body: str
    tags: list
    links: list
    importance: float
    access_count: int
    date: str

def _slug(title: str) -> str:
    return re.sub(r"[^a-z0-9-]", "-", title.lower()).strip("-")

def _file_hash(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def get_collection() -> chromadb.Collection:
    client = chromadb.PersistentClient(path=DB_PATH)
    return client.get_or_create_collection(name="vault", metadata={"hnsw:space": "cosine"})


def write_note(title: str, body: str, tags: list, links: list) -> Path:
    """TODO: build YAML frontmatter, format body with ## Related backlinks, write to VAULT."""
    # Frontmatter: date, tags, importance=0.5, access_count=0
    # Related section: '\n'.join(f'- [[{_slug(l)}]]' for l in links)
    # Log: log.info("memory write: %s", title)
    pass


def _parse(path: Path) -> Note | None:
    """Helper: parse frontmatter and body from a vault note."""
    raw = path.read_text(encoding="utf-8")
    match = re.match(r"^---\n(.*?)\n---\n(.*)$", raw, re.DOTALL)
    if not match:
        return None
    meta = yaml.safe_load(match.group(1))
    body = match.group(2).strip()
    return Note(
        title=path.stem, body=body,
        tags=meta.get("tags", []), links=re.findall(r"\[\[([^\]]+)\]\]", body),
        importance=meta.get("importance", 0.5),
        access_count=meta.get("access_count", 0),
        date=str(meta.get("date", "")),
    )


def read_note(title: str) -> Note | None:
    """TODO: read note, increment access_count in file, return Note or None."""
    pass


def list_notes() -> list:
    return [n for n in (_parse(p) for p in VAULT.glob("*.md")) if n]


def sync_embeddings():
    """TODO: upsert new/changed notes, delete stale entries. Use _file_hash for change detection."""
    # existing = {m["slug"]: m.get("hash") for m in collection.get(include=["metadatas"])["metadatas"] or []}
    # seen = set()
    # for each path in VAULT.glob("*.md"): check hash, upsert if changed
    # delete slugs in existing - seen
    pass


def search_similar(query: str, k: int = 4) -> list:
    """TODO: query ChromaDB, return Notes by looking up slugs from ids."""
    pass


print("Memory functions defined")

<details>
<summary>💡 Stuck? Reveal the memory solutions</summary>

```python
def write_note(title: str, body: str, tags: list, links: list) -> Path:
    frontmatter = {"date": date.today().isoformat(), "tags": tags, "importance": 0.5, "access_count": 0}
    related = "\n".join(f"- [[{_slug(l)}]]" for l in links)
    content = f"---\n{yaml.safe_dump(frontmatter)}---\n\n{body}\n\n## Related\n{related}\n"
    path = VAULT / f"{_slug(title)}.md"
    path.write_text(content, encoding="utf-8")
    log.info("memory write: %s", title)
    return path

def read_note(title: str) -> Note | None:
    path = VAULT / f"{_slug(title)}.md"
    if not path.exists():
        return None
    note = _parse(path)
    if note:
        raw = path.read_text(encoding="utf-8")
        new_count = note.access_count + 1
        raw = re.sub(r"access_count: \d+", f"access_count: {new_count}", raw, count=1)
        path.write_text(raw, encoding="utf-8")
        note.access_count = new_count
        log.info("memory read: %s (access %d)", title, new_count)
    return note

def sync_embeddings():
    collection = get_collection()
    existing = {m["slug"]: m.get("hash") for m in
                (collection.get(include=["metadatas"])["metadatas"] or [])}
    seen = set()
    for path in VAULT.glob("*.md"):
        slug, h = path.stem, _file_hash(path)
        seen.add(slug)
        if existing.get(slug) != h:
            note = _parse(path)
            if note:
                collection.upsert(ids=[slug], documents=[note.body],
                                   metadatas=[{"slug": slug, "hash": h, "title": note.title}])
                log.info("index upsert: %s", slug)
    stale = set(existing) - seen
    if stale:
        collection.delete(ids=list(stale))

def search_similar(query: str, k: int = 4) -> list:
    collection = get_collection()
    if collection.count() == 0:
        return []
    hits = collection.query(query_texts=[query], n_results=min(k, collection.count()))
    return [n for n in (read_note(slug) for slug in hits["ids"][0]) if n]
```

</details>

## Step 5 — Build the consolidator (TODO)

Implement `score_importance(note)` and `consolidate(threshold=0.2)`.

- Recency: `max(0, 1 - age_days / 180)` (decays to 0 after 180 days)
- Access: `min(note.access_count / 10, 1.0)` (normalised to 0-1)
- Final score: `0.4 * recency + 0.6 * access`

In [ ]:
def score_importance(note: Note) -> float:
    """TODO: return 0.4 * recency + 0.6 * access."""
    pass


def consolidate(importance_threshold: float = 0.2) -> dict:
    """TODO: score all notes, archive those below threshold, call sync_embeddings().
    Returns {"archived": int, "total": int}."""
    pass


print("Consolidator defined")

<details>
<summary>💡 Stuck? Reveal the consolidator solution</summary>

```python
def score_importance(note: Note) -> float:
    try:
        age_days = (date.today() - date.fromisoformat(note.date)).days
    except ValueError:
        age_days = 180
    recency = max(0.0, 1.0 - age_days / 180)
    access = min(note.access_count / 10, 1.0)
    return 0.4 * recency + 0.6 * access

def consolidate(importance_threshold: float = 0.2) -> dict:
    notes = list_notes()
    archived = 0
    for note in notes:
        if score_importance(note) < importance_threshold:
            src = VAULT / f"{note.title}.md"
            if src.exists():
                src.rename(ARCHIVE / src.name)
                archived += 1
                log.info("memory archive: %s", note.title)
    sync_embeddings()
    return {"archived": archived, "total": len(notes)}
```

</details>

## Step 6 — The research agent (TODO)

Complete Step 3 in `run_research()`: extract tags and related topics from the synthesis, call `write_note()`, and call `sync_embeddings()`.

In [ ]:
import json
import litellm

litellm.drop_params = True

SYSTEM_PROMPT = """You are a research assistant that builds a persistent knowledge base.
When given a topic:
1. Synthesise what is known into clear, well-structured markdown
2. Identify 2-4 related concepts that should be cross-linked
3. Return your response as JSON:
   {"synthesis": "...", "tags": ["tag1", "tag2"], "related": ["concept1", "concept2"]}
"""

MAX_MEMORY_RESULTS = 4


def run_research(topic: str) -> str:
    log.info(f"Researching: {topic}")

    # Step 1: Check memory first
    memory_hits = search_similar(topic, k=MAX_MEMORY_RESULTS)
    if memory_hits:
        log.info(f"Memory hit: found {len(memory_hits)} relevant notes")
        memory_context = "\n\n".join(n.body for n in memory_hits)
        memory_section = f"Existing memory:\n{memory_context}\n\n"
    else:
        log.info("No memory hit — proceeding to synthesis")
        memory_section = "No existing memory on this topic.\n\n"

    # Step 2: Synthesise
    response = litellm.completion(
        model=MODEL, max_tokens=1024,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{memory_section}Research topic: {topic}\n\nSynthesise and structure your knowledge."}
        ]
    )
    synthesis = response.choices[0].message.content

    # Step 3: Write note back to vault
    # TODO: parse synthesis JSON to extract body, tags, links
    # TODO: call write_note(title=topic, body=body, tags=tags, links=links)
    # TODO: call sync_embeddings()
    # TODO: return body
    return synthesis


print("Research agent defined ✓")

<details>
<summary>💡 Stuck? Reveal Step 3 solution</summary>

```python
    # Step 3: Write note back to vault
    try:
        parsed = json.loads(synthesis)
        body = parsed["synthesis"]
        tags = parsed.get("tags", [])
        links = parsed.get("related", [])
    except (json.JSONDecodeError, KeyError):
        body, tags, links = synthesis, [], []
        log.warning("synthesis was not valid JSON; storing raw text")

    if not links:  # backlink fallback: link to retrieved notes
        links = [n.title for n in memory_hits[:2]]

    write_note(title=topic, body=body, tags=tags, links=links)
    sync_embeddings()
    return body
```

The backlink fallback guarantees the hard constraint (every note has ≥1 backlink) even when the model returns raw text.

</details>

## Step 7 — Test your agent

In [ ]:
# Session 1: write two notes
result1 = run_research("transformer attention mechanism")
print(f"Session 1a: {len(result1)} chars written\n")

result2 = run_research("positional encoding in transformers")
print(f"Session 1b: {len(result2)} chars written")

notes = list_notes()
print(f"\nVault has {len(notes)} notes")
for n in notes:
    assert "[[" in n.body, f"Note '{n.title}' has no backlinks!"
print("All notes have backlinks ✓")

In [ ]:
# Session 2: agent should find memory from session 1
result3 = run_research("how does the transformer architecture work?")
print("Session 2 result:")
print(result3[:300], "...")
# Check the logs — you should see "Memory hit: found X relevant notes"

In [ ]:
# Consolidation test
stats = consolidate(importance_threshold=0.8)  # high threshold to force archiving
print(f"Archived: {stats['archived']}/{stats['total']} notes")
assert "archived" in stats
assert "total" in stats
print("Consolidation test passed ✓")

## Step 8 — Experiments

### Experiment A: Memory vs no-memory
Comment out the `search_similar` call in `run_research`. Re-research the same topics. Compare token usage and output quality.

### Experiment B: Stale memory
Manually edit a note's `date:` frontmatter to one year ago. Run `consolidate(threshold=0.3)`. Does it get archived?

### Experiment C: Swap the model
Change `MODEL` to `gpt-4o-mini`. Does the JSON output format stay consistent? What happens when it doesn't?

### Experiment D: Vault inspection
Open `vault/notes/` and read the markdown files. Are the backlinks meaningful? Does the vault read like a human-authored wiki?

In [ ]:
# Scratch cell — use for experiments


## Step 9 — Reflection

1. In one sentence: what is the vault and what is ChromaDB, and why do you need both?
2. What happens if you delete the vault but keep ChromaDB? What if you do the reverse?
3. Why does `sync_embeddings` use hash comparison instead of re-embedding everything?
4. What breaks first when the model stops returning valid JSON?

*(Double-click to edit)*

1. 
2. 
3. 
4. 